# Data Ingestion Method 3: Auto Loader

## Overview
Auto Loader is Databricks' serverless, incremental data ingestion service. It automatically detects and ingests new data files as they arrive, handling schema inference and evolution.

## Key Characteristics
- **Best for**: Streaming data, real-time ingestion, continuously arriving files
- **Automatic Detection**: Uses cloud storage notifications
- **Schema Evolution**: Handles schema changes automatically
- **Idempotent**: Processes each file exactly once
- **Serverless**: No cluster management needed


## Prerequisites

1. Databricks workspace with Auto Loader enabled
2. Access to cloud storage (S3, Azure Blob Storage, GCS)
3. Source data directory configured
4. Databricks Runtime with Structured Streaming support


## Step 1: Using Auto Loader with PySpark

Auto Loader is typically used with PySpark for streaming operations. We'll ingest real-time event data.

In [ ]:
# Import necessary libraries
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

In [ ]:
# Define schema for event data
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("page_url", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("device_type", StringType(), True)
])

print("Schema defined for Auto Loader - Event Data")

## Step 2: Basic Auto Loader Configuration

Configure Auto Loader to ingest data from a cloud storage path.


In [ ]:
# Configure Auto Loader source path for event data
source_path = "/Volumes/data_analysis/sample_data/"
checkpoint_path = "/tmp/autoloader_checkpoint/events/"
table_name = "events_autoloader"

print(f"Source: {source_path}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Table: {table_name}")

In [ ]:
# Create Auto Loader stream for CSV event files
df_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", checkpoint_path) \
    .schema(schema) \
    .option("header", "true") \
    .load(source_path)

print("Auto Loader stream created successfully for event data")

## Step 3: Add Metadata and Write to Delta Table


In [ ]:
# Add ingestion metadata
df_with_metadata = df_stream.withColumn(
    "ingestion_timestamp", 
    current_timestamp()
).withColumn(
    "source_file",
    spark.sql("input_file_name()")  # Captures the source file name
)

print("Metadata added to stream")

In [ ]:
# Write stream to Delta table with checkpoint
streaming_query = df_with_metadata.writeStream \
    .format("delta") \
    .mode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .table(table_name)

print(f"Streaming query started: {streaming_query.id}")
print(f"Status: {streaming_query.status}")

## Step 4: Monitor Auto Loader Progress


In [ ]:
# Check streaming query status
print(f"Query ID: {streaming_query.id}")
print(f"Status: {streaming_query.status}")
print(f"Recent Progress: {streaming_query.recentProgress}")

In [ ]:
-- Check data in the table
SELECT COUNT(*) as total_rows FROM events_autoloader

In [ ]:
-- View sample data with metadata
SELECT * FROM events_autoloader LIMIT 5

## Step 5: Auto Loader with Schema Evolution

Auto Loader can handle schema changes automatically.


In [ ]:
# Auto Loader with schema evolution enabled
checkpoint_evolving = "/tmp/autoloader_checkpoint/events_evolving/"
table_evolving = "events_autoloader_evolving"

df_evolving = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", checkpoint_evolving) \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .schema(schema) \
    .option("header", "true") \
    .load(source_path)

print("Auto Loader stream with schema evolution created")

In [ ]:
# Write with schema evolution
streaming_evolving = df_evolving \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .writeStream \
    .format("delta") \
    .mode("append") \
    .option("checkpointLocation", checkpoint_evolving) \
    .option("mergeSchema", "true") \
    .table(table_evolving)

print(f"Schema evolution enabled for {table_evolving}")

## Step 6: Auto Loader with Notifications

For better performance, use cloud storage notifications (file-based triggering).


In [ ]:
# Auto Loader with notification triggering (for production)
# Requires SQS queue setup for S3 or Event Grid for Azure

# Example configuration (comment: adjust for your cloud provider)
notification_config = {
    # AWS S3 with SQS notifications
    "cloudFiles.format": "csv",
    "cloudFiles.schemaLocation": checkpoint_path,
    "cloudFiles.includeExistingFiles": "false",  # Only new files
    "cloudFiles.useNotifications": "true",  # Enable notifications
    # "cloudFiles.notificationQueueArn": "arn:aws:sqs:...",  # SQS queue
}

print("Notification-based Auto Loader configuration ready")
print("Uncomment and add your notification queue ARN for production use")

## Step 7: Stop Streaming Query


In [ ]:
# Stop the streaming queries when done
# Uncomment to stop
# streaming_query.stop()
# streaming_evolving.stop()

print("Streaming queries can be stopped with .stop() method")

## Advantages of Auto Loader

✅ **Automatic Detection**: Detects new files without polling

✅ **Schema Evolution**: Automatically handles schema changes

✅ **Exactly-Once Processing**: Guarantees no duplicate processing

✅ **Serverless**: No infrastructure to manage

✅ **Real-time Ingestion**: Processes data as soon as it arrives

✅ **Cost-Efficient**: Scales automatically

✅ **Failure Recovery**: Automatic retry and recovery


## Disadvantages of Auto Loader

❌ **Complexity**: More setup than CTAS or COPY INTO

❌ **Streaming Overhead**: Requires streaming infrastructure

❌ **Cost**: Streaming queries incur cluster costs

❌ **Notifications Setup**: Requires cloud-specific configuration (SQS, Event Grid, etc.)


## Use Cases for Auto Loader

1. **Real-time Data Pipeline**: Continuous data ingestion from cloud storage
2. **IoT Data Processing**: Streaming sensor data ingestion
3. **Log Aggregation**: Collecting logs from multiple sources
4. **Event Processing**: Processing event streams from event hubs
5. **Evolving Schemas**: Data sources with frequently changing schemas
6. **Data Lake Ingestion**: Continuous ingestion into data lakes


## Auto Loader vs COPY INTO vs CTAS

| Aspect | CTAS | COPY INTO | Auto Loader |
|--------|------|-----------|-------------|
| **Setup Complexity** | Low | Medium | High |
| **Incremental** | No | Yes | Yes |
| **Schema Evolution** | No | No | Yes |
| **Real-time** | No | No | Yes |
| **Idempotent** | No | Yes | Yes |
| **Best For** | Initial load | Batch load | Streaming |
| **Cost** | Low | Low | Medium |


## Summary

Auto Loader is ideal for:
- Real-time data ingestion
- Streaming data pipelines
- Automatic schema evolution
- Serverless data processing

Choose based on your use case:
- **One-time load**: Use CTAS
- **Regular batch load**: Use COPY INTO
- **Real-time/Streaming**: Use Auto Loader
